# Bloom Filter Encoding

## Objective
Convert identifiers into privacy-preserving representations using Bloom filters.

In [1]:
# =================== BOOTSTRAP CELL ===================
# Standard setup for all notebooks
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# from src.config.variables import COVARIATES

# ========================================================
# Optional for warnings and nicer plots
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

import sys
from pathlib import Path

# ========================================================
# 1️⃣ Ensure project root is in Python path
# Adjust this if your notebooks are nested deeper
PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
# 2️⃣ Import helper to load paths
from src.utils.helpers import load_paths

# ========================================================
# 3️⃣ Load paths from config.yaml (works regardless of notebook location)
paths = load_paths()

# ========================================================
# 4️⃣ Optionally, print paths to confirm
for key, value in paths.items():
    print(f"{key}: {value}")

# ========================================================
# 5️⃣ Now you can use these paths in your notebook:
# Example:
SYNTHETIC_DATA_DIR = paths['SYNTHETIC_DATA_DIR']
PROCESSED_DATA_DIR = paths['PROCESSED_DATA_DIR']
# FIG_DIR = paths['FIG_DIR']

# ========================================================

BASE_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters
DATA_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\data
OUT_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\model_output
FIG_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\visualization
MODEL_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\model_output\statsmodels
NOTEBOOKS_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\pprl_notebooks
NOTEBOOKS_EXECUTED_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\notebooks_executed
RAW_DATA_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\data\raw
PROCESSED_DATA_DIR: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\data

### Load data

In [2]:
import pandas as pd
from bitarray import bitarray
import hashlib

# Load
df_A = pd.read_pickle(PROCESSED_DATA_DIR / "df_A_preprocessed.pkl")
df_B = pd.read_pickle(PROCESSED_DATA_DIR / "df_B_preprocessed.pkl")

### Q-grams

introduces error tolerance - they allow fuzzy matching

In [3]:
def get_qgrams(text, q=2):
    if not isinstance(text, str):
        text = str(text) if text is not None else ""
    text = text.replace(" ", "")
    return [text[i:i+q] for i in range(len(text) - q + 1)]

# get_qgrams("otieno otieno2010-10-15")

Q-grams allow approximate matching by capturing substrings.

### Bloom Filter

In [4]:
from bitarray import bitarray
import hashlib

def bloom_encode(text, size=1024, num_hashes=5):
    # Creating a blank memory
    bf = bitarray(size)
    bf.setall(0)

    qgrams = get_qgrams(text) # Breaking the word into pieces (overlapping pieces) - like tokenization

    for qg in qgrams:
        for i in range(num_hashes):  # Create multiple versions of the same qgram
            h = int(hashlib.sha1((qg + str(i)).encode()).hexdigest(), 16) # Turn each version into a number
            bf[h % size] = 1         # Pick a spot in the list and mark the spot

    return bf
    '''
    Combine: "DA" + "0" = "DA0", Encode: b'DA0' - byte (DAO), Hash: SHA1 gives us a fingerprint, 
    Hexdigest: "a94a8fe5ccb19ba61c4c0873d391e987982fbbd3", 
    Convert: This becomes the number 763275293847209384720938472093847209384720938
    '''
    # 

#### Apply encoding

"john1990" → 010010101010... -> what is shared instead of raw identifiers

In [5]:
df_A["bloom"] = df_A["combined"].apply(bloom_encode)
df_B["bloom"] = df_B["combined"].apply(bloom_encode)

Each record is now represented as a bit array instead of raw identifiers.

### Save


In [6]:
df_A.to_pickle(PROCESSED_DATA_DIR / "df_A_encoded.pkl")
df_B.to_pickle(PROCESSED_DATA_DIR / "df_B_encoded.pkl")